# CAMELS: Plot Meteorological Time Series
***

**_Autor:_** Chus Casado Rodríguez<br>
**_Date:_** 27-08-2026<br>

**Introduction:**<br>
This script creates HTML plots to visualize the daily meteorological time series for the selected dataset: CERRA, EMO1 or ROCIO-IBEB.

In [ ]:
from tqdm.auto import tqdm
import pandas as pd
import geopandas as gpd

from ocab.config import Config
from ocab.plots.meteo import plot_meteo_timeseries
import ocab.meteorology as METEO
import ocab.variables as VARS


## Configuration


In [ ]:
cfg = Config('config_CAMELS_v200.yml')
meteo_dataset = 'ROCIO-IBEB'

# paths
path_in = cfg.path_dataset / 'preprocessing' / 'timeseries' / 'meteo' / meteo_dataset
path_plots = path_in / 'plots'
path_plots.mkdir(exist_ok=True, parents=True)

# variables = {
#     'ta_mean': 'temp_degC', 
#     'pr_mean': 'precip_mm', 
#     'e0_mean': 'pet_mm',
# }

## Create plots

In [13]:
# load stations
stations = gpd.read_file(cfg.path_gis / 'stations.geojson').set_index('id')

# process timeseries for each station
for ID in tqdm(stations.index, desc='stations'):

    # meteo timeseries
    try:
        meteo = pd.read_parquet(path_in / f'{ID:04d}.parquet').loc[ID]
        meteo.rename(columns=VARS.RENAME, inplace=True, errors='ignore')
        
        # correct dates
        if meteo_dataset in METEO.OFFSET_HOURS:
            meteo.index = meteo.index + pd.Timedelta(hours=METEO.OFFSET_HOURS[meteo_dataset])
        meteo.index.name = 'date'
        meteo.index = pd.to_datetime(meteo.index)
    except Exception as e:
        print(f'Error loading meteo timeseries for station {ID:04d}: {e}')
        continue
    
    # extract attributes and time series
    attrs = stations.loc[ID]

    # create time series plot
    try:
        title = '{0} - {1} - River {2} ({3})'.format(
            ID, 
            attrs['name'].title(), 
            attrs['river'].title(), 
            attrs['basin'].title()
        )
        fig = plot_meteo_timeseries(meteo, title=title, save=True)
        # fig.write_html(path_plots / f'{ID:04d}.html')
        fig.write_html(f'{ID:04d}.html')
    except:
        print(f"The plot for time series {ID} couldn't be created")

    break

stations:   0%|          | 0/1116 [00:00<?, ?it/s]

In [11]:
meteo.columns

Index(['precip_mm', 'precip_max_mm', 'precip_std_mm', 'precipitation_cv',
       'precip_frac', 'temp_degC', 'temp_min_degC', 'temp_max_degC', 'pet_mm',
       'temp_dtr_degC'],
      dtype='str')

In [12]:
meteo[meteo.columns.intersection(VARS.RENAME.values())]

,precip_mm,precip_max_mm,precip_std_mm,precip_frac,temp_degC,temp_min_degC,temp_max_degC,pet_mm,temp_dtr_degC
date,,,,,,,,,
1979-01-02,4.415312,7.110000,1.573089,1.000000,5.455715,3.219894,7.691535,0.546914,4.471641
1979-01-03,1.582518,2.710000,0.550142,0.794872,1.372901,-1.680883,4.426685,0.529623,6.107568
1979-01-04,11.396941,13.950000,1.149268,1.000000,1.962394,-2.485124,6.409912,0.662255,8.895036
1979-01-05,2.336875,4.410000,1.099442,0.897436,5.481660,1.760044,9.203277,0.717767,7.443233
1979-01-06,1.414531,4.870000,1.257312,0.487179,4.418800,0.982009,7.855590,0.661955,6.873581
...,...,...,...,...,...,...,...,...,...
2022-12-28,0.125073,0.290000,0.063764,0.000000,11.258266,8.122462,14.394070,0.795008,6.271608
2022-12-29,0.150419,0.440000,0.134269,0.000000,10.725921,7.146810,14.305031,0.834599,7.158221
2022-12-30,14.050661,16.719999,1.321244,1.000000,11.358708,9.293902,13.423515,0.645603,4.129613
